# SRQ4 — ViT Baseline Comparison

Trains EfficientFormer-L1 under the same two-phase transfer learning protocol used for all CNN experiments.
All hyperparameters are loaded from the CNN baseline grid search — identical protocol, no separate ViT tuning.
This is a deliberate methodological choice: the comparison tests architectural paradigm under shared constraints,
not per-architecture optimisation.

**Protocol:**
- Phase 1 `param_mode='head'`: transformer encoder frozen, only classifier head warms up
- Phase 2 `param_mode='all'`: full fine-tuning with gradient clipping (max_norm=1.0)
- Phase 2 patience increased to 5 (vs 3 for CNNs) — ViTs stabilise slower after unfreezing
- All hyperparameters loaded from `grid-search-results/optimal_config.csv` (same as CNN baseline)
- 5-fold stratified CV; best fold selected by val F1 for test evaluation
- **Section 6:** Final held-out test set evaluation — run once only after all CV decisions are made


## 1 · Paths & Imports

In [1]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[1]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "vit-comparison-results"
WEIGHTS_DIR   = RESULTS_DIR / "weights"
CURVES_DIR    = RESULTS_DIR / "training-curves"
PLOTS_DIR     = RESULTS_DIR / "plots"

# Upstream experiment directories (read-only)
ARCH_EVAL_RESULTS_DIR     = PROJECT_ROOT / "data" / "experiments" / "arch-eval-results"
HEAD_ABLATION_RESULTS_DIR = PROJECT_ROOT / "data" / "experiments" / "head-ablation-results"
GRID_SEARCH_RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "grid-search-results"

for d in [RESULTS_DIR, WEIGHTS_DIR, CURVES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Weights dir  : {WEIGHTS_DIR}")

Project root : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir     : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Results dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results
Weights dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\weights


In [2]:
import csv
import time
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import f1_score

import src.scripts.data      as data
import src.scripts.models    as models
import src.scripts.trainer   as trainer
import src.scripts.utils     as utils
import src.scripts.evaluator as evaluator

utils.set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Random seed set to 42 for Python, NumPy, and PyTorch
Device: cpu


## 2 · Data — Outer Split (Identical to All Other Experiments)

Fixed seed 42 outer 80/20 stratified split. All training and model selection operates within `X_trainval`.
The held-out test set is set aside until final evaluation in Section 8.

In [3]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

print(f"\nSRQ4 operates on {len(X_trainval)} train+val samples.")
print("Held-out test set is NOT used until Section 8 (final evaluation).")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201

SRQ4 operates on 1321 train+val samples.
Held-out test set is NOT used until Section 8 (final evaluation).


## 3 · Configuration

All hyperparameters loaded from `grid-search-results/optimal_config.csv` — identical to the CNN baseline.
Phase 1 parameters are fixed a priori (same as all CNN experiments).
Phase 2 LR and weight decay are the CNN-optimal values from the grid search.
Phase 2 patience is increased from 3 to 5 to allow ViT stabilisation after unfreezing.
Gradient clipping (max_norm=1.0) is applied in Phase 2 — standard practice for ViT fine-tuning.


In [4]:
# ── Load hyperparameters from CNN baseline grid search ───────────────────────
OPTIMAL_CONFIG_FILE = GRID_SEARCH_RESULTS_DIR / "optimal_config.csv"
if not OPTIMAL_CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"optimal_config.csv not found at {OPTIMAL_CONFIG_FILE}\n"
        "Run the grid search notebook first."
    )

optimal = pd.read_csv(OPTIMAL_CONFIG_FILE).iloc[0]
LR_PHASE1         = float(optimal["lr_phase1"])
WD_PHASE1         = float(optimal["wd_phase1"])
BEST_LR_PHASE2    = float(optimal["lr_phase2"])
BEST_WEIGHT_DECAY = float(optimal["weight_decay"])

# ── Fixed protocol parameters ─────────────────────────────────────────────────
SEED           = 42
BATCH_SIZE     = 32
WINNING_HEAD   = "linear"
P1_EPOCHS      = 20
P1_PATIENCE    = 4
P2_EPOCHS      = 15
P2_PATIENCE    = 5       # increased from 3: ViTs stabilise slower than ResNets
GRAD_CLIP_NORM = 1.0     # standard for ViT fine-tuning; no-op for Phase 1
ARCHITECTURE   = "efficientformer"

print(f"Loaded from : {OPTIMAL_CONFIG_FILE.name}")
print(f"  lr_phase1    = {LR_PHASE1:.0e}")
print(f"  wd_phase1    = {WD_PHASE1}")
print(f"  lr_phase2    = {BEST_LR_PHASE2:.0e}")
print(f"  weight_decay = {BEST_WEIGHT_DECAY:.0e}")
print(f"")
print(f"Architecture : {ARCHITECTURE}")
print(f"Head         : {WINNING_HEAD}")
print(f"Phase 1      : epochs={P1_EPOCHS}  patience={P1_PATIENCE}")
print(f"Phase 2      : epochs={P2_EPOCHS}  patience={P2_PATIENCE}  grad_clip={GRAD_CLIP_NORM}")
print(f"Note         : Phase 2 hyperparameters are CNN-optimal values — same protocol, no separate ViT tuning")


Loaded from : optimal_config.csv
  lr_phase1    = 1e-03
  wd_phase1    = 0.0
  lr_phase2    = 1e-04
  weight_decay = 1e-04

Architecture : efficientformer
Head         : linear
Phase 1      : epochs=20  patience=4
Phase 2      : epochs=15  patience=5  grad_clip=1.0
Note         : Phase 2 hyperparameters are CNN-optimal values — same protocol, no separate ViT tuning


## 4 · 5-Fold CV Training — EfficientFormer

Trains EfficientFormer under the CNN-optimal config using 5-fold stratified CV.
Weights saved per fold. Results saved fold-by-fold; safe to interrupt and resume.


In [5]:
N_SPLITS     = 5
RESULTS_FILE = RESULTS_DIR / "vit_cv_results.csv"
CV_FIELDNAMES = [
    "architecture", "lr_phase1", "wd_phase1", "lr_phase2", "weight_decay",
    "fold", "val_acc", "val_loss", "val_f1",
    "p1_epochs_run", "p2_epochs_run",
    "weights_path", "timestamp", "error"
]

total_cv_runs = N_SPLITS
print(f"EfficientFormer × {N_SPLITS} folds = {total_cv_runs} runs")
print(f"lr_phase2 : {BEST_LR_PHASE2:.0e}  weight_decay : {BEST_WEIGHT_DECAY:.0e}")
print(f"Results   → {RESULTS_FILE}")
print(f"Weights   → {WEIGHTS_DIR}/{ARCHITECTURE}/fold_<n>.pt")

EfficientFormer × 5 folds = 5 runs
lr_phase2 : 1e-04  weight_decay : 1e-04
Results   → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\vit_cv_results.csv
Weights   → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\weights/efficientformer/fold_<n>.pt


In [6]:
completed_cv = utils.load_completed_runs(
    RESULTS_FILE,
    [("architecture", str), ("fold", int)]
)
cv_run_number = len(completed_cv)
print(f"Device: {DEVICE}")
print(f"{N_SPLITS} folds — {cv_run_number} already completed\n")

for fold_idx in range(N_SPLITS):

    run_key = (ARCHITECTURE, fold_idx)
    if run_key in completed_cv:
        print(f"SKIP  fold={fold_idx+1}/{N_SPLITS}")
        continue

    utils.set_seed(SEED)
    cv_run_number += 1
    print(f"\n{'='*65}")
    print(f"  Run {cv_run_number}/{total_cv_runs}  |  fold={fold_idx+1}/{N_SPLITS}")
    print(f"{'='*65}")

    error_msg = ""
    val_f1 = val_acc = val_loss = float("nan")
    p1_epochs_run = p2_epochs_run = 0
    wpath = utils.weights_path_for(WEIGHTS_DIR, ARCHITECTURE, fold_idx)

    try:
        train_loader, val_loader = data.get_fold_loaders(
            X_trainval, y_trainval,
            fold_idx=fold_idx,
            train_transform=train_transform,
            test_transform=test_transform,
            n_splits=N_SPLITS,
            batch_size=BATCH_SIZE,
            SEED=SEED
        )

        model = models.get_model(architecture=ARCHITECTURE, head=WINNING_HEAD)

        run_configs = {
            "phase1": {
                "num_epochs"  : P1_EPOCHS,
                "lr"          : LR_PHASE1,
                "parameters"  : "head",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": WD_PHASE1,
            },
            "phase2": {
                "num_epochs"  : P2_EPOCHS,
                "lr"          : BEST_LR_PHASE2,
                "parameters"  : "all",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": BEST_WEIGHT_DECAY,
            },
        }

        # Phase 1
        losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase1",
            train_configs=run_configs,
            early_stopping_patience=P1_PATIENCE
        )
        p1_epochs_run = len(val_accs_p1)

        utils.plot(
            losses_p1, accs_p1,
            config_name=f"phase1_fold{fold_idx}",
            val_losses=val_losses_p1, val_accuracies=val_accs_p1,
            model_name=ARCHITECTURE,
            save_dir=str(CURVES_DIR),
            show=False
        )

        # Phase 2 with gradient clipping
        losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase2",
            train_configs=run_configs,
            early_stopping_patience=P2_PATIENCE,
            grad_clip_norm=GRAD_CLIP_NORM
        )
        p2_epochs_run = len(val_accs_p2)

        utils.plot(
            losses_p2, accs_p2,
            config_name=f"phase2_fold{fold_idx}",
            val_losses=val_losses_p2, val_accuracies=val_accs_p2,
            model_name=ARCHITECTURE,
            save_dir=str(CURVES_DIR),
            show=False
        )

        val_loss = val_losses_p2[-1]
        val_acc  = val_accs_p2[-1]

        # Val F1
        model.eval()
        y_true_val, y_pred_val = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs = imgs.to(DEVICE)
                probs = torch.sigmoid(model(imgs)).cpu().numpy().flatten()
                preds = (probs >= 0.5).astype(int)
                y_true_val.extend(lbls.numpy().astype(int))
                y_pred_val.extend(preds)
        val_f1 = f1_score(y_true_val, y_pred_val)

        utils.save_weights(model, wpath)
        print(f"  val_f1={val_f1:.4f}  val_acc={val_acc:.4f}  val_loss={val_loss:.4f}  "
              f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")
        print(f"  Weights → {wpath}")

    except Exception as e:
        error_msg = str(e)
        print(f"  ERROR: {error_msg}")
        traceback.print_exc()

    utils.append_result(RESULTS_FILE, CV_FIELDNAMES, {
        "architecture" : ARCHITECTURE,
        "lr_phase1"    : LR_PHASE1,
        "wd_phase1"    : WD_PHASE1,
        "lr_phase2"    : BEST_LR_PHASE2,
        "weight_decay" : BEST_WEIGHT_DECAY,
        "fold"         : fold_idx,
        "val_acc"      : round(val_acc, 4) if not np.isnan(val_acc) else "",
        "val_loss"     : round(val_loss, 4) if not np.isnan(val_loss) else "",
        "val_f1"       : round(val_f1, 4) if not np.isnan(val_f1) else "",
        "p1_epochs_run": p1_epochs_run,
        "p2_epochs_run": p2_epochs_run,
        "weights_path" : str(wpath),
        "timestamp"    : datetime.now().isoformat(timespec="seconds"),
        "error"        : error_msg,
    })

print(f"\n{'='*65}")
print("CV TRAINING COMPLETE")
print(f"Results → {RESULTS_FILE}")

Device: cpu
5 folds — 0 already completed

Random seed set to 42 for Python, NumPy, and PyTorch

  Run 1/5  |  fold=1/5
get_fold_loaders()>>> Fold 1/5 — Train: 1056,  Val: 265
get_model()>>> architecture='efficientformer'  head='linear'
[phase1] Epoch 1/20 - Train Loss: 0.6211 - Train Acc: 0.6629 - Val Loss: 0.5860 - Val Acc: 0.6679
[phase1] Epoch 2/20 - Train Loss: 0.5052 - Train Acc: 0.7803 - Val Loss: 0.4824 - Val Acc: 0.8075
[phase1] Epoch 3/20 - Train Loss: 0.4528 - Train Acc: 0.8011 - Val Loss: 0.4222 - Val Acc: 0.8113
[phase1] Epoch 4/20 - Train Loss: 0.4243 - Train Acc: 0.8248 - Val Loss: 0.4019 - Val Acc: 0.8113
[phase1] Epoch 5/20 - Train Loss: 0.4057 - Train Acc: 0.8305 - Val Loss: 0.3885 - Val Acc: 0.8226
[phase1] Epoch 6/20 - Train Loss: 0.3817 - Train Acc: 0.8456 - Val Loss: 0.3647 - Val Acc: 0.8340
[phase1] Epoch 7/20 - Train Loss: 0.3698 - Train Acc: 0.8485 - Val Loss: 0.3579 - Val Acc: 0.8340
[phase1] Epoch 8/20 - Train Loss: 0.3744 - Train Acc: 0.8409 - Val Loss: 0.34

get_model()>>> architecture='efficientformer'  head='linear'
[phase1] Epoch 1/20 - Train Loss: 0.6177 - Train Acc: 0.6660 - Val Loss: 0.5899 - Val Acc: 0.6629
[phase1] Epoch 2/20 - Train Loss: 0.4957 - Train Acc: 0.7843 - Val Loss: 0.4983 - Val Acc: 0.7841
[phase1] Epoch 3/20 - Train Loss: 0.4516 - Train Acc: 0.8155 - Val Loss: 0.4472 - Val Acc: 0.8144
[phase1] Epoch 4/20 - Train Loss: 0.4203 - Train Acc: 0.8297 - Val Loss: 0.4262 - Val Acc: 0.8295
[phase1] Epoch 5/20 - Train Loss: 0.4084 - Train Acc: 0.8212 - Val Loss: 0.4169 - Val Acc: 0.8447
[phase1] Epoch 6/20 - Train Loss: 0.3883 - Train Acc: 0.8467 - Val Loss: 0.4072 - Val Acc: 0.8447
[phase1] Epoch 7/20 - Train Loss: 0.3700 - Train Acc: 0.8562 - Val Loss: 0.4073 - Val Acc: 0.8409
[phase1] Epoch 8/20 - Train Loss: 0.3745 - Train Acc: 0.8392 - Val Loss: 0.3926 - Val Acc: 0.8485
[phase1] Epoch 9/20 - Train Loss: 0.3597 - Train Acc: 0.8562 - Val Loss: 0.3951 - Val Acc: 0.8409
[phase1] Epoch 10/20 - Train Loss: 0.3461 - Train Acc: 0.

## 5 · CV Results Summary


In [7]:
df_cv = pd.read_csv(RESULTS_FILE)
df_cv["error"] = df_cv["error"].fillna("")
df_ok   = df_cv[df_cv["error"] == ""].copy()
df_fail = df_cv[df_cv["error"] != ""].copy()

for col in ["val_f1", "val_acc", "val_loss"]:
    df_ok[col] = df_ok[col].astype(float)

print(f"Successful runs : {len(df_ok)} / {total_cv_runs}")
print(f"Failed runs     : {len(df_fail)}")
if len(df_fail):
    print(df_fail[["fold", "error"]].to_string(index=False))

Successful runs : 5 / 5
Failed runs     : 0


In [9]:
if arch_eval_csv.exists():
    df_arch = pd.read_csv(arch_eval_csv)
    df_arch["error"] = df_arch["error"].fillna("")
    df_arch = df_arch[df_arch["error"] == ""]

    if df_arch.empty:
        print("No successful CNN runs found in arch_eval_results.csv — comparison skipped")
    else:
        best_cnn_mean = df_arch.groupby("architecture")["val_f1"].mean().max()
        best_cnn_arch = df_arch.groupby("architecture")["val_f1"].mean().idxmax()
        print(f"\n  Best CNN CV mean F1 : {best_cnn_mean:.4f} ({best_cnn_arch})")
        print(f"  ViT CV mean F1      : {df_ok.val_f1.mean():.4f}")
        print(f"  Δ (ViT − CNN)       : {df_ok.val_f1.mean() - best_cnn_mean:+.4f}")


  Best CNN CV mean F1 : 0.9254 (cbam_end)
  ViT CV mean F1      : 0.8848
  Δ (ViT − CNN)       : -0.0406


In [10]:
# ── Fold selection — best val F1 ──────────────────────────────────────────────
if not df_ok.empty:
    best_fold_row = df_ok.sort_values("val_f1", ascending=False).iloc[0]
    SELECTED_FOLD = int(best_fold_row["fold"])
    print(f"Selected fold {SELECTED_FOLD+1}  "
          f"(val_f1={best_fold_row.val_f1:.4f}  val_loss={best_fold_row.val_loss:.4f})")

Selected fold 3  (val_f1=0.9442  val_loss=0.2380)


## 6 · Final Test Set Evaluation — SRQ4

Loads the best fold weights for EfficientFormer and the best CNN from arch-eval.
Both evaluated once on the held-out test set. Run once only after all model selection decisions are made.


In [11]:
test_loader = data.get_test_loader(X_test, y_test, test_transform, batch_size=BATCH_SIZE)
print(f"Device: {DEVICE}")

get_test_loader()>>> Test loader ready — 331 samples
Device: cpu


In [12]:
test_results = {}

# ── Evaluate EfficientFormer ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Test evaluation: {ARCHITECTURE}  (fold {SELECTED_FOLD+1})")
print(f"{'='*60}")

vit_weights = utils.weights_path_for(WEIGHTS_DIR, ARCHITECTURE, SELECTED_FOLD)

if not vit_weights.exists():
    print(f"  SKIP — weights not found at {vit_weights}")
else:
    utils.set_seed(SEED)
    vit_model = models.get_model(architecture=ARCHITECTURE, head=WINNING_HEAD)
    vit_model = utils.load_weights(vit_model, vit_weights, device=DEVICE)

    acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
        model=vit_model, test_loader=test_loader, device=DEVICE
    )
    test_results[ARCHITECTURE] = {
        "test_f1": round(f1, 4), "test_auc": round(auc, 4),
        "test_ece": round(ece, 4), "test_acc": round(acc, 4),
        "test_prec": round(prec, 4), "test_rec": round(rec, 4),
        "conf_matrix": conf.tolist(),
        "lr_phase2": BEST_LR_PHASE2, "weight_decay": BEST_WEIGHT_DECAY,
    }


  Test evaluation: efficientformer  (fold 3)
Random seed set to 42 for Python, NumPy, and PyTorch
get_model()>>> architecture='efficientformer'  head='linear'
load_weights()>>> Model loaded successfully and set to evaluation mode.
Accuracy  : 0.9517
Precision : 0.9914
Recall    : 0.8846
F1        : 0.9350
AUC-ROC   : 0.9864
ECE       : 0.0474  (lower = better calibrated; 0 = perfect)
Confusion Matrix:
 [[200   1]
 [ 15 115]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      1.00      0.96       201
           1       0.99      0.88      0.93       130

    accuracy                           0.95       331
   macro avg       0.96      0.94      0.95       331
weighted avg       0.95      0.95      0.95       331



In [13]:
# ── Evaluate best CNN from arch-eval ─────────────────────────────────────────
arch_eval_csv = ARCH_EVAL_RESULTS_DIR / "arch_eval_results.csv"

if arch_eval_csv.exists():
    df_arch = pd.read_csv(arch_eval_csv)
    df_arch = df_arch[df_arch["error"] == ""]

    # Best CNN arch by mean val F1
    best_cnn_arch = df_arch.groupby("architecture")["val_f1"].mean().idxmax()
    # Best fold for that arch
    cnn_best_fold = int(
        df_arch[df_arch["architecture"] == best_cnn_arch]
        .sort_values("val_f1", ascending=False)
        .iloc[0]["fold"]
    )
    cnn_weights = ARCH_EVAL_RESULTS_DIR / "weights" / best_cnn_arch / f"fold_{cnn_best_fold}.pt"

    print(f"\n{'='*60}")
    print(f"  Test evaluation: {best_cnn_arch} (best CNN, fold {cnn_best_fold+1})")
    print(f"{'='*60}")

    if not cnn_weights.exists():
        print(f"  SKIP — weights not found at {cnn_weights}")
    else:
        utils.set_seed(SEED)
        cnn_model = models.get_model(architecture=best_cnn_arch, head=WINNING_HEAD)
        cnn_model = utils.load_weights(cnn_model, cnn_weights, device=DEVICE)

        acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
            model=cnn_model, test_loader=test_loader, device=DEVICE
        )
        test_results[best_cnn_arch] = {
            "test_f1": round(f1, 4), "test_auc": round(auc, 4),
            "test_ece": round(ece, 4), "test_acc": round(acc, 4),
            "test_prec": round(prec, 4), "test_rec": round(rec, 4),
            "conf_matrix": conf.tolist(),
            "lr_phase2": None, "weight_decay": None,
        }
else:
    print("arch-eval results not found — CNN comparison skipped")

ValueError: attempt to get argmax of an empty sequence

In [ ]:
# ── SRQ4 comparison table ─────────────────────────────────────────────────────
df_test = pd.DataFrame(test_results).T
df_test = df_test.drop(columns=["conf_matrix"])
for col in ["test_f1", "test_auc", "test_ece", "test_acc", "test_prec", "test_rec"]:
    df_test[col] = df_test[col].astype(float)

print("SRQ4 — Final Test Results\n")
print(f"{'Model':<30} {'F1':>7} {'AUC':>7} {'ECE':>7} {'Acc':>7}")
print("-" * 58)
for arch, row in df_test.iterrows():
    print(f"  {arch:<28} {row.test_f1:>7.4f} {row.test_auc:>7.4f} "
          f"{row.test_ece:>7.4f} {row.test_acc:>7.4f}")

# Delta
if ARCHITECTURE in df_test.index and best_cnn_arch in df_test.index:
    delta_f1  = df_test.loc[ARCHITECTURE, "test_f1"]  - df_test.loc[best_cnn_arch, "test_f1"]
    delta_auc = df_test.loc[ARCHITECTURE, "test_auc"] - df_test.loc[best_cnn_arch, "test_auc"]
    delta_ece = df_test.loc[ARCHITECTURE, "test_ece"] - df_test.loc[best_cnn_arch, "test_ece"]
    print(f"\n  Δ ({ARCHITECTURE} − {best_cnn_arch}):")
    print(f"    ΔF1  = {delta_f1:+.4f}")
    print(f"    ΔAUC = {delta_auc:+.4f}")
    print(f"    ΔECE = {delta_ece:+.4f}  (negative = better calibrated)")

In [14]:
# ── Save test results ─────────────────────────────────────────────────────────
TEST_RESULTS_FILE = RESULTS_DIR / "srq4_test_results.csv"
df_test.to_csv(TEST_RESULTS_FILE)
print(f"Test results saved → {TEST_RESULTS_FILE}")

NameError: name 'df_test' is not defined

In [15]:
# ── Visualisation ─────────────────────────────────────────────────────────────
if not df_test.empty:
    COLOUR_MAP = {
        ARCHITECTURE: "#4C72B0",   # blue — ViT
    }
    if best_cnn_arch in df_test.index:
        COLOUR_MAP[best_cnn_arch] = "#DD8452"   # amber — CNN

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("SRQ4 — CNN vs EfficientFormer", fontsize=13, fontweight="bold")

    plot_order = df_test.sort_values("test_f1", ascending=True)
    colours    = [COLOUR_MAP.get(a, "#888888") for a in plot_order.index]
    y_pos      = np.arange(len(plot_order))

    # Panel 1: F1 bar chart
    ax = axes[0]
    ax.barh(y_pos, plot_order["test_f1"], color=colours,
            alpha=0.85, height=0.5, edgecolor="white", linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_order.index, fontsize=10)
    ax.set_xlabel("Test F1", fontsize=11)
    ax.set_title("Test F1", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    f1_vals = plot_order["test_f1"].values
    pad = max((f1_vals.max() - f1_vals.min()) * 0.4, 0.02)
    ax.set_xlim(max(0, f1_vals.min() - pad), min(1, f1_vals.max() + pad))
    for y, v in zip(y_pos, f1_vals):
        ax.text(v + 0.002, y, f"{v:.4f}", va="center", fontsize=9)

    # Panel 2: AUC and ECE grouped bars
    ax2   = axes[1]
    x     = np.arange(len(df_test))
    w     = 0.35
    ax2.bar(x - w/2, df_test["test_auc"], w, label="AUC-ROC",
            color="#4C72B0", alpha=0.82)
    ax2.bar(x + w/2, df_test["test_ece"], w, label="ECE (↓ better)",
            color="#DD8452", alpha=0.82)
    ax2.set_xticks(x)
    ax2.set_xticklabels(list(df_test.index), rotation=15, ha="right", fontsize=9)
    ax2.set_ylabel("Score", fontsize=11)
    ax2.set_title("AUC-ROC & ECE", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    utils.save_fig(fig, PLOTS_DIR, "srq4_test_results", formats=("svg",))
    plt.show()

NameError: name 'df_test' is not defined